In [28]:
from collections import Counter

In [29]:
#I want to check if user_bank_payment is a valid subset of bank, if it is remove the necessary payment from the bank list
#first i want to do this without a function and build from there

#Now add logic to pay with properties if you can

money_charged = 4

bank  = [5,3]
property_value = [2,2,3,3]

user_bank_payment = [3] #subset of bank
user_property_payment = [2]

#############################

bank_counts = Counter(bank)
user_payment_counts = Counter(user_bank_payment)

total_payment = sum(user_bank_payment)
if total_payment < money_charged:
    print("Payment is less than required amount.")
else:
    for user_card, user_count in user_payment_counts.items():
        if user_count > bank_counts.get(user_card, 0):
            print(f"Payment includes {user_card} which is not in bank or exceeds available count.")
            break
        elif user_count <= bank_counts.get(user_card, 0):
            #remove the necessary payment from the bank list
            bank.remove(user_card)
    print("Payment Processed.")


Payment is less than required amount.


## Final Payment Processing Code

In [30]:
#Payment processing code v2

#Split the payment from the removing from bank and property list logic

def remove_payment_from_list(payment_list, payment_counts):
    for payment_card, payment_count in payment_counts.items():
        for _ in range(payment_count):
            payment_list.remove(payment_card)


def process_payment(money_charged,
                    bank,
                    properties,
                    user_bank_payment,
                    user_property_payment):
    
    bank_counts = Counter(bank)
    properties_counts = Counter(properties)
    user_bank_payment_counts = Counter(user_bank_payment)
    user_property_payment_counts = Counter(user_property_payment)

    available_total = sum(bank) + sum(properties)
    offered_total = sum(user_bank_payment) + sum(user_property_payment)

    #Validate user is paying with cards they have
    for payment_card, payment_count in user_bank_payment_counts.items():
        if payment_count > bank_counts.get(payment_card, 0):
            print(f"Payment includes {payment_card} which is not in bank or exceeds available count.")
            raise ValueError("Invalid bank payment.")
        
    for payment_card, payment_count in user_property_payment_counts.items():
        if payment_count > properties_counts.get(payment_card, 0):
            print(f"Payment includes {payment_card} which is not in properties or exceeds available count.")
            raise ValueError("Invalid property payment.")
        
    paid_all_bank = (user_bank_payment_counts == bank_counts)
    paid_all_props = (user_property_payment_counts == properties_counts)

    if offered_total >= money_charged:
        print("Valid Payment")
        remove_payment_from_list(bank, user_bank_payment_counts)
        remove_payment_from_list(properties, user_property_payment_counts)
    elif (available_total < money_charged) and (offered_total == available_total) and paid_all_bank and paid_all_props:
        print("Valid Payment with all available cards")
        remove_payment_from_list(bank, user_bank_payment_counts)
        remove_payment_from_list(properties, user_property_payment_counts)
    else:
        print("Payment is less than required amount.")
        raise ValueError("Insufficient payment.")
    
    return(bank,properties)


In [32]:
#Testing the functions

money_charged = 21

bank  = [5,4,3]
properties = [2,2]

user_bank_payment = [5,4,3]
user_property_payment = [2,2]

bank, properties = process_payment(money_charged,
                bank,
                properties,
                user_bank_payment,
                user_property_payment)

bank

Valid Payment with all available cards


[]

## Structure of Application

## JSON card structure

In [33]:
import os, json

base_path = os.getcwd()

all_cards_json_path = os.path.join(base_path, "all_cards.json")

all_cards_json = json.load(open(all_cards_json_path, "r"))
cards_path = os.path.join(base_path,"backend" ,"cards", "base")

for card in all_cards_json:
    card_filename = f"{card['name']}.json"
    card_path = os.path.join(cards_path, card_filename)
    with open(card_path, "w") as f:
        json.dump(card, f, indent=4)


### Pydantic Models:

JSON -> Validate with Raw Pydantic Models -> CardDef for engine gameplay

In [37]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Any, Dict, List, Optional, Literal

class RawEffect(BaseModel):
    type: str
    model_config = ConfigDict(extra="allow")  # keep other keys

class RawCard(BaseModel):
    id: str
    name: str
    type: str                 # your JSON uses "type"
    count: int
    bank_value: int

    colors: Optional[List[str]] = None
    property_group: Optional[str] = None
    effect: Optional[RawEffect] = None

    bankable: Optional[bool] = None
    image_url: Optional[str] = None
    restrictions: Optional[Dict[str, Any]] = None
    note: Optional[str] = None

    model_config = ConfigDict(extra="allow")  # keep any future fields

class PlayDef(BaseModel):
    effect: str
    params: Dict[str, Any] = Field(default_factory=dict)

class CardDef(BaseModel):
    id: str
    name: str
    kind: Literal["money","property","property_wild","rent","action"]
    copies: int
    money_value: int
    colors: Optional[List[str]] = None
    play: Optional[PlayDef] = None
    meta: Dict[str, Any] = Field(default_factory=dict)



In [45]:
#Convert RawCard to CardDef
def raw_to_carddef(raw: RawCard) -> Optional[CardDef]:
    kind_map = {
        "money": "money",
        "property": "property",
        "property_wild": "property_wild",
        "rent": "rent",
        "action": "action"
    }
    
    kind = kind_map.get(raw.type)
    if not kind:
        return None  # Unknown type
    
    play_def = None
    if raw.effect:
        play_def = PlayDef(effect=raw.effect.type, params={k: v for k, v in raw.effect.model_dump().items() if k != "type"})
    
    card_def = CardDef(
        id=raw.id,
        name=raw.name,
        kind=kind,
        copies=raw.count,
        money_value=raw.bank_value,
        colors=raw.colors,
        play=play_def,
        meta={
            "property_group": raw.property_group,
            "bankable": raw.bankable,
            "image_url": raw.image_url,
            "restrictions": raw.restrictions,
            "note": raw.note
        }
    )
    
    return card_def

In [46]:
#Load a single card definition from JSON file


def load_card_defs(path: str) -> list[CardDef]:
    data = json.loads(open(path, "r", encoding="utf-8").read())
    raw_cards = RawCard.model_validate(data)
    #raw_cards = [RawCard.model_validate(x) for x in data]
    #defs = [raw_to_carddef(rc) for rc in raw_cards]
    return raw_cards


load_card_defs(os.path.join(cards_path, "Illinois Avenue.json"))

RawCard(id='prop_red_illinois_avenue', name='Illinois Avenue', type='property', count=1, bank_value=3, colors=None, property_group='red', effect=None, bankable=True, image_url='https://monopolydealrules.com/photos/cards/red-property-card.png', restrictions=None, note=None, set_size=3, rent_by_count=[2, 3, 6])

#### Build the Main Card Catalog Function

In [47]:
#Load all cards

from pathlib import Path
import json
from typing import Dict, List

def load_card_defs_from_dir(cards_dir: str) -> Dict[str, CardDef]:
    catalog: Dict[str,CardDef] = {}

    for path in Path(cards_dir).glob("*.json"):
        raw_data = json.loads(path.read_text(encoding="utf-8"))

        # Normalize to a list
        if isinstance(raw_data, list):
            raw_cards = [RawCard.model_validate(item) for item in raw_data]
        else:
            raw_cards = [RawCard.model_validate(raw_data)]

        for rc in raw_cards:
            cd = raw_to_carddef(rc)
            if cd is None:
                continue
            if cd.id in catalog:
                raise ValueError(f"Duplicate card ID found: {cd.id}")
            catalog[cd.id] = cd

    return catalog



## GameState, PlayerState, and DeckState

In [48]:
from typing import Dict, List, Optional
from pydantic import BaseModel, Field


class DeckState(BaseModel):
    draw_pile: List[str] = Field(default_factory = list)
    discard_pile: List[str] = Field(default_factory = list)

class PlayerState(BaseModel):
    id: str
    hand: List[str] = Field(default_factory=list)        # card ids
    bank: List[str] = Field(default_factory=list)        # card ids
    properties: Dict[str, List[str]] = Field(default_factory=dict)  # color -> card ids

class GameState(BaseModel):
    id: str
    players: Dict[str, PlayerState]
    deck: DeckState
    current_player_id: Optional[str] = None
    turn_number: int = 1




## Create a Deck using Pydantic Models we built, Create a new game, and Draw Cards functions


In [ ]:
import random
from typing import Dict, List

def build_deck(catalog: Dict[str, CardDef], seed = 42) -> List[str]:
    deck: List[str] = []

    #catalog is key: card id, value: CardDef
    for cd in catalog.values():
        deck.extend([cd.id] * cd.copies)
    
    #Shuffle deck with seed
    random.seed(seed)
    random.shuffle(deck)

    #Check if deck is 110 cards long

    if len(deck) != 106: #110 - 4 helper cards
        
        raise ValueError(f"Deck size is {len(deck)}, expected 110.")
    
    return deck


#Test this function out

catalog = load_card_defs_from_dir(cards_path)
deck = build_deck(catalog, seed=123)



['action_debt_collector',
 'money_3m',
 'action_debt_collector',
 'multicolor_rent',
 'prop_railroad_b_and_o_railroad',
 'prop_green_pacific_avenue',
 'prop_red_indiana_avenue',
 'action_pass_go',
 'action_forced_deal',
 'wild_red_yellow',
 'wild_pink_orange',
 'action_pass_go',
 'prop_dark_blue_boardwalk',
 'prop_dark_blue_park_place',
 'action_its_my_birthday',
 'action_house',
 'money_3m',
 'money_4m',
 'rent_red_yellow',
 'money_5m',
 'action_just_say_no',
 'wild_multicolor_any',
 'wild_railroad_utility',
 'action_forced_deal',
 'action_pass_go',
 'money_1m',
 'prop_utility_electric_company',
 'rent_railroad_utility',
 'prop_red_kentucky_avenue',
 'action_double_the_rent',
 'action_pass_go',
 'rent_pink_orange',
 'money_3m',
 'wild_dark_blue_green',
 'wild_multicolor_any',
 'prop_green_pennsylvania_avenue',
 'money_4m',
 'action_its_my_birthday',
 'action_just_say_no',
 'multicolor_rent',
 'rent_light_blue_brown',
 'prop_green_north_carolina_avenue',
 'rent_light_blue_brown',
 'act